# 🩺 Fine-tuning LoRA — Modèle médical expérimental
**Projet TechCorp IA — Mission R&D (filière IA)**

⚠️ **Avertissements** :
- Modèle **expérimental**, **non destiné à la production** ni à un usage clinique réel.
- Ne remplace **jamais** l'avis d'un professionnel de santé. Validation médicale obligatoire avant tout usage.
- Dataset public `ruslanmv/ai-medical-chatbot` (dialogues Patient/Docteur). Respect RGPD / anonymisation pour toute donnée réelle.

**Runtime** : Colab → `Exécution` → `Modifier le type d'exécution` → **GPU (T4)**.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi

## 2. Installer les dépendances
*(si une erreur de version apparaît : `Exécution` → `Redémarrer la session`, puis relancer à partir d'ici.)*

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets
import torch
print("CUDA dispo :", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 3. Charger le dataset et inspecter sa structure
*Toujours regarder les colonnes reelles avant de formater.*

In [ ]:
from datasets import load_dataset

raw = load_dataset("ruslanmv/ai-medical-chatbot", split="train")
print("Colonnes :", raw.column_names)
print("Taille totale :", len(raw))
print("\n--- Exemple ---")
ex = raw[0]
for k, v in ex.items():
    print(f"[{k}] {str(v)[:200]}")

## 4. Préparer les données (sous-échantillon expérimental)
On prend un sous-ensemble (2000 dialogues) pour un entrainement rapide. Question = message du patient, Reponse = message du docteur, au format de prompt **Phi-3**.

In [ ]:
import re

N_SAMPLES = 2000           # augmente si tu as le temps
MAX_ANSWER_CHARS = 1200    # coupe les reponses trop longues

cols = raw.column_names
q_col = "Patient" if "Patient" in cols else ("Description" if "Description" in cols else cols[0])
a_col = "Doctor" if "Doctor" in cols else cols[-1]
print(f"Question = '{q_col}'  |  Reponse = '{a_col}'")

subset = raw.shuffle(seed=42).select(range(min(N_SAMPLES, len(raw))))

def clean(t):
    return re.sub(r"\s+", " ", str(t)).strip()

def to_text(ex):
    q = clean(ex[q_col])
    a = clean(ex[a_col])[:MAX_ANSWER_CHARS]
    return {"text": f"<|user|>\n{q}<|end|>\n<|assistant|>\n{a}<|end|>"}

ds = subset.map(to_text, remove_columns=cols)
ds = ds.filter(lambda e: len(e["text"]) > 60)   # retire les exemples vides/trop courts
print("Exemples conserves :", len(ds))
print("\n--- Apercu formate ---\n", ds[0]["text"][:400])

## 5. Charger le modèle de base en 4-bit + configurer LoRA
Modele : **Phi-3.5-mini-instruct** (coherent avec le modele financier du projet).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
import torch

BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",   # T4 n'a pas flash-attention
)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.1, bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6. Tokeniser

In [ ]:
def tokenize(batch):
    out = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=512)
    out["labels"] = out["input_ids"].copy()
    return out

tokenized = ds.map(tokenize, batched=True, remove_columns=["text"])
print(tokenized)

## 7. Entraîner (LoRA)
Quelques centaines de steps suffisent pour une demo experimentale. La **loss** est journalisee tous les 10 steps.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir="./medical_lora",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=20,
    logging_steps=10,
    save_strategy="no",
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

## 8. Courbe de loss (métrique d'entraînement)

In [ ]:
import matplotlib.pyplot as plt

hist = [h for h in trainer.state.log_history if "loss" in h]
steps = [h["step"] for h in hist]
loss  = [h["loss"] for h in hist]

plt.figure(figsize=(8,4))
plt.plot(steps, loss, marker="o")
plt.title("Loss d'entrainement - LoRA medical")
plt.xlabel("step"); plt.ylabel("loss"); plt.grid(True)
plt.show()

print(f"Loss initiale : {loss[0]:.3f}  ->  finale : {loss[-1]:.3f}")
print(f"Steps : {steps[-1]}  |  Epochs : {args.num_train_epochs}")

## 9. Tester le modèle fine-tuné

In [ ]:
def ask(question, max_new_tokens=200):
    prompt = f"<|user|>\n{question}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    model.eval()
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, do_sample=True, top_p=0.9,
            repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id,
            use_cache=False,            # <-- contourne le bug DynamicCache.seen_tokens
        )
    txt = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return txt.strip()

for q in [
    "What are common symptoms of seasonal allergies?",
    "How can I manage mild dehydration at home?",
    "What does a high blood pressure reading mean?",
]:
    print("Q:", q)
    print("A:", ask(q), "\n", "-"*60)

## 10. Sauvegarder l'adapter + les métriques (livrables)

In [ ]:
import json

model.save_pretrained("./medical_lora_adapter")
tokenizer.save_pretrained("./medical_lora_adapter")

metrics = {
    "base_model": BASE_MODEL,
    "samples": len(tokenized),
    "epochs": args.num_train_epochs,
    "final_loss": round(loss[-1], 4),
    "loss_history": hist,
}
with open("./medical_lora_adapter/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Adapter + metrics sauvegardes dans ./medical_lora_adapter")
!cd medical_lora_adapter && zip -qr ../medical_lora_adapter.zip . && echo "zip pret"
from google.colab import files
files.download("medical_lora_adapter.zip")

## 📦 Ce que tu rends
- **Lien Colab partage** (Partager -> acces en lecture).
- **Metriques** : courbe de loss (cellule 8) + `metrics.json` (loss initiale/finale, epochs, steps).
- **Quelques exemples** de reponses du modele (cellule 9).
- Rappel : modele **experimental, non deploye** — conforme au sujet.